# Cocopila: Pandas Data Agent Pipeline — Kaggle Execution Notebook 🥥📊⚡

Notebook này cài đặt **Ollama**, nạp mô hình **qwen2.5-coder:1.5b** (hoặc tùy chọn 7B) và thực thi toàn bộ **LangGraph Agent Pipeline** (5 Nodes + Vòng lặp Reflection) một cách đơn giản, gọn nhẹ.

In [ ]:
# 1. Cài đặt các gói phụ thuộc dự án (Không cần vLLM giúp tăng tốc độ cài đặt và tránh xung đột)
print("📥 Đang cài đặt dependencies...")
!pip install -q \
    langgraph>=0.2.0 \
    langchain-core>=0.3.0 \
    langchain-openai>=0.2.0 \
    pyyaml>=6.0 \
    json-repair>=0.30.0 \
    openpyxl>=3.1.0 \
    tabulate>=0.9.0 \
    thefuzz>=0.22.0

# 2. Tải và cài đặt Ollama
print("📥 Đang tải và cài đặt Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("✅ Cài đặt phụ thuộc và Ollama thành công!")

In [ ]:
import subprocess
import time
import requests
import os

# 1. Khởi động Ollama Server chạy nền
print("🚀 Đang khởi động Ollama Server...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 2. Chờ Ollama Server sẵn sàng
print("⏳ Chờ Ollama Server khởi động...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/")
        if r.status_code == 200:
            print("✅ Ollama Server đã sẵn sàng tại port 11434!")
            break
    except Exception:
        time.sleep(1)
else:
    print("❌ Lỗi: Ollama Server không thể khởi động.")

# 3. Pull model (Mặc định sử dụng qwen2.5-coder:1.5b gọn nhẹ để demo nhanh)
MODEL_NAME = "qwen2.5-coder:1.5b"
print(f"📥 Đang tải mô hình {MODEL_NAME} từ Ollama registry...")
subprocess.run(["ollama", "pull", MODEL_NAME])
print(f"✅ Đã tải thành công mô hình {MODEL_NAME}!")

In [ ]:
# 3. Thiết lập Biến Môi trường & Chạy Thử Kết nối (Sanity Check)
os.environ["MODEL_NAME"] = MODEL_NAME
os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"] = "ollama"

from langchain_openai import ChatOpenAI

# Gọi thử model để xác minh hoạt động
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_base="http://localhost:11434/v1",
    openai_api_key="ollama",
    temperature=0,
)

response = llm.invoke("Xin chào! Kiểm tra kết nối từ LangChain?")
print("🤖 Phản hồi thử nghiệm từ Ollama:")
print(response.content)

In [ ]:
# 4. THỰC THI TOÀN BỘ COCOPILA LANGGRAPH PIPELINE
import sys
sys.path.insert(0, "/kaggle/working/cocopila")

from pipeline.src.graph import create_cocopila_graph

# Khởi tạo LangGraph Agent Graph (sử dụng cấu hình Ollama từ biến môi trường)
agent = create_cocopila_graph()

# Truy vấn mẫu
user_query = "Tính tổng doanh thu và số lượng đơn hàng theo dòng sản phẩm từ file sample_sales"

initial_state = {
    "user_query": user_query,
    "status": "pending",
    "retry_count": 0,
    "node_latencies": {}
}

print(f"🚀 Bắt đầu chạy Agent Pipeline cho câu hỏi: '{user_query}'...")
final_state = agent.invoke(initial_state)

print("\n=================== KẾT QUẢ XỬ LÝ AGENT ===================")
print("📌 Trạng thái (Status):", final_state.get("status"))
print("📌 Phân tích Intent (Parsed Query):", final_state.get("parsed_query"))
print("📌 File dữ liệu khớp (Matched File):", final_state.get("matched_table_path"))
print("📌 Ánh xạ Cột (Column Mapping):", final_state.get("column_mapping"))
print("📌 Mã Pandas Sinh Ra (Generated Code):\n", final_state.get("generated_code"))
print("📌 Kết Quả Thực Thi (Execution Result):\n", final_state.get("execution_result"))
print("📌 Thời gian xử lý từng Node (Latencies):", final_state.get("node_latencies"))
print("===========================================================")